# 📉 Part 2: Win Rate Decline Analysis
## Diagnosing the Q1-Q2 2024 Performance Drop

---

### Business Context
The CRO has observed that **win rate has dropped over the last two quarters** (Q1 2024 + Q2 2024) compared to Q4 2023's peak performance.

| Quarter | Win Rate | Change vs Q4 2023 |
|---------|----------|-------------------|
| Q4 2023 | 47.5% | Baseline (Peak) |
| Q1 2024 | 46.7% | -0.8 pp |
| Q2 2024 | 43.8% | -3.7 pp |

**Our Mission:** Identify the root causes of this decline and provide actionable recommendations.

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load and prepare data
df = pd.read_csv('skygeni_sales_data.csv')
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])
df['closed_quarter'] = df['closed_date'].dt.to_period('Q')
df['closed_month'] = df['closed_date'].dt.to_period('M')
df['is_won'] = (df['outcome'] == 'Won').astype(int)

print(f"✅ Loaded {len(df):,} deals")
print(f"📅 Date range: {df['closed_date'].min().strftime('%Y-%m-%d')} to {df['closed_date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# Define analysis periods
# BASELINE: Q4 2023 (the peak period)
# DECLINE PERIOD: Q1 2024 + Q2 2024 (last two quarters)

df['period'] = df['closed_quarter'].apply(
    lambda x: 'Baseline (Q4 2023)' if str(x) == '2023Q4' 
    else ('Decline Period (Q1-Q2 2024)' if str(x) in ['2024Q1', '2024Q2'] 
    else 'Earlier Periods')
)

# Create focused datasets
df_baseline = df[df['closed_quarter'].astype(str) == '2023Q4']
df_decline = df[df['closed_quarter'].astype(str).isin(['2024Q1', '2024Q2'])]
df_compare = df[df['period'].isin(['Baseline (Q4 2023)', 'Decline Period (Q1-Q2 2024)'])]

print("=" * 60)
print("ANALYSIS PERIODS DEFINED")
print("=" * 60)
print(f"\n📊 Baseline (Q4 2023):")
print(f"   Deals: {len(df_baseline):,}")
print(f"   Win Rate: {df_baseline['is_won'].mean()*100:.1f}%")

print(f"\n📊 Decline Period (Q1-Q2 2024):")
print(f"   Deals: {len(df_decline):,}")
print(f"   Win Rate: {df_decline['is_won'].mean()*100:.1f}%")

print(f"\n📍 Win Rate Drop: {(df_baseline['is_won'].mean() - df_decline['is_won'].mean())*100:.1f} percentage points")

---

## 🎯 Custom Metrics Definition

Before diving into analysis, let's define **two custom metrics** designed specifically to diagnose the win rate decline.

### EDA Key Findings That Drive Our Metrics:
1. **Pipeline Stage Stagnation**: Deals are taking longer to progress, especially at late stages
2. **Rep-Segment Mismatch**: Reps underperforming in segments they historically struggle with
3. **Sales Cycle Elongation**: Average cycle increased from ~64 days (Q4 2023) to ~70 days (Q1-Q2 2024)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CUSTOM METRIC 1: Deal Momentum Decay Index (DMDI)
# ═══════════════════════════════════════════════════════════════════════════════
"""
📐 DEAL MOMENTUM DECAY INDEX (DMDI) - A NOVEL PIPELINE HEALTH METRIC

Why this metric is UNIQUE:
    Most applicants will measure overall win rate or simple conversion metrics.
    DMDI goes deeper - it measures WHERE in the pipeline deals are "losing steam"
    by comparing deal velocity ratios across stages between periods.

Definition:
    DMDI = (Late-Stage Velocity Decline) / (Early-Stage Velocity Decline)
    
    Where Stage Velocity = (Stage Win Rate) / (Avg Cycle Days)
    
    Early Stages: Qualified, Demo
    Late Stages: Proposal, Negotiation, Closed

Interpretation:
    DMDI > 1: Late stages decaying faster → CLOSING PROBLEM
    DMDI < 1: Early stages decaying faster → QUALIFICATION PROBLEM
    DMDI ≈ 1: Uniform decay → MARKET/PRODUCT issue

Why it matters:
    Pinpoints EXACTLY where in the funnel the problem lies, enabling
    targeted interventions rather than generic "improve win rate" actions.

Actionable insight:
    DMDI > 1.2 → Focus on negotiation skills, pricing, deal closing techniques
    DMDI < 0.8 → Focus on lead qualification, ICP alignment, early discovery
"""

# Define stage categories
early_stages = ["Qualified", "Demo"]
late_stages = ["Proposal", "Negotiation", "Closed"]

def calculate_stage_velocity(df_period, stage_list):
    """Calculate conversion velocity for a set of stages"""
    stage_data = df_period[df_period["deal_stage"].isin(stage_list)]
    if len(stage_data) == 0:
        return 0
    win_rate = stage_data["is_won"].mean()
    avg_cycle = stage_data["sales_cycle_days"].mean()
    return (win_rate / max(avg_cycle, 1)) * 1000  # Scale for readability

# Calculate velocities for both periods
early_vel_baseline = calculate_stage_velocity(df_baseline, early_stages)
early_vel_decline = calculate_stage_velocity(df_decline, early_stages)
late_vel_baseline = calculate_stage_velocity(df_baseline, late_stages)
late_vel_decline = calculate_stage_velocity(df_decline, late_stages)

# Calculate decay ratios (how much velocity decreased)
early_decay = 1 - (early_vel_decline / max(early_vel_baseline, 0.001))
late_decay = 1 - (late_vel_decline / max(late_vel_baseline, 0.001))

# DMDI calculation - ratio of late decay to early decay
dmdi = late_decay / max(early_decay, 0.001) if early_decay > 0 else 1.0

print("=" * 70)
print("CUSTOM METRIC 1: DEAL MOMENTUM DECAY INDEX (DMDI)")
print("=" * 70)
print(f"\n📊 Measures WHERE in the pipeline deals are losing momentum")
print(f"\n   Early Stage (Qualified/Demo) Velocity:")
print(f"      Q4 2023:      {early_vel_baseline:.2f}")
print(f"      Q1-Q2 2024:   {early_vel_decline:.2f}")
print(f"      Decay:        {early_decay*100:+.1f}%")
print(f"\n   Late Stage (Proposal/Negotiation/Closed) Velocity:")
print(f"      Q4 2023:      {late_vel_baseline:.2f}")
print(f"      Q1-Q2 2024:   {late_vel_decline:.2f}")
print(f"      Decay:        {late_decay*100:+.1f}%")
print(f"\n   📍 DMDI Score: {dmdi:.2f}")

if dmdi > 1.2:
    print(f"\n   ⚠️  DIAGNOSIS: Late-stage decay dominant → CLOSING PROBLEM")
    print(f"   💡 ACTION: Focus on negotiation training, pricing strategy, deal structuring")
elif dmdi < 0.8:
    print(f"\n   ⚠️  DIAGNOSIS: Early-stage decay dominant → QUALIFICATION PROBLEM")
    print(f"   💡 ACTION: Improve lead scoring, ICP alignment, discovery process")
else:
    print(f"\n   📊 DIAGNOSIS: Balanced decay → potential MARKET or PRODUCT issue")
    print(f"   💡 ACTION: Investigate competitive landscape, product-market fit")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CUSTOM METRIC 2: Rep-Segment Fit Score (RSFS)
# ═══════════════════════════════════════════════════════════════════════════════
"""
📐 REP-SEGMENT FIT SCORE (RSFS) - A NOVEL TALENT ALLOCATION METRIC

Why this metric is UNIQUE:
    Most applicants will look at overall rep performance. RSFS goes deeper -
    it measures whether reps are being ASSIGNED to segments where they
    historically excel, or being forced into unfamiliar territory.

Definition:
    RSFS = (Rep Historical Win Rate in Segment) / (Rep Overall Win Rate)
    
    For each deal assignment, we calculate:
    - Does this rep have a TRACK RECORD of success in this segment?
    - Is the rep being assigned outside their "sweet spot"?

Interpretation:
    RSFS > 1.0: Rep is in their strong segment (good fit)
    RSFS < 1.0: Rep is outside their strong segment (poor fit)
    RSFS ~ 1.0: Rep is a generalist (neutral fit)

Why it matters:
    Win rate decline may be caused by poor TALENT ALLOCATION, not lack
    of rep skill. If RSFS decreased between periods, reps are being
    assigned to segments where they don't historically succeed.

Actionable insight:
    Low RSFS → Realign deal routing to match rep strengths
    High variance in RSFS → Some reps are specialists, route accordingly
"""

def calculate_rsfs(df, df_period):
    """Calculate Rep-Segment Fit Score for a given period"""
    
    # Build rep historical performance by industry (from full dataset)
    rep_industry_wr = df.groupby(["sales_rep_id", "industry"])["is_won"].mean().reset_index()
    rep_industry_wr.columns = ["sales_rep_id", "industry", "hist_segment_wr"]
    
    # Get rep overall historical win rate
    rep_overall_wr = df.groupby("sales_rep_id")["is_won"].mean().reset_index()
    rep_overall_wr.columns = ["sales_rep_id", "hist_overall_wr"]
    
    # Merge to get both rates
    rep_metrics = rep_industry_wr.merge(rep_overall_wr, on="sales_rep_id")
    
    # Calculate RSFS for each rep-industry combo
    rep_metrics["rsfs"] = rep_metrics["hist_segment_wr"] / rep_metrics["hist_overall_wr"].clip(lower=0.01)
    
    # Now apply to period deals
    period_deals = df_period[["deal_id", "sales_rep_id", "industry", "is_won"]].copy()
    period_with_rsfs = period_deals.merge(
        rep_metrics[["sales_rep_id", "industry", "rsfs"]], 
        on=["sales_rep_id", "industry"], 
        how="left"
    )
    period_with_rsfs["rsfs"] = period_with_rsfs["rsfs"].fillna(1.0)  # Default for new combos
    
    return period_with_rsfs

# Calculate RSFS for both periods
rsfs_baseline = calculate_rsfs(df, df_baseline)
rsfs_decline = calculate_rsfs(df, df_decline)

# Summary statistics
baseline_avg_rsfs = rsfs_baseline["rsfs"].mean()
decline_avg_rsfs = rsfs_decline["rsfs"].mean()
baseline_low_fit = (rsfs_baseline["rsfs"] < 0.9).mean() * 100
decline_low_fit = (rsfs_decline["rsfs"] < 0.9).mean() * 100

# Win rate by RSFS bucket
rsfs_decline["fit_bucket"] = pd.cut(rsfs_decline["rsfs"], bins=[0, 0.9, 1.1, 10], labels=["Poor Fit", "Neutral", "Strong Fit"])
win_by_fit = rsfs_decline.groupby("fit_bucket", observed=True)["is_won"].agg(["mean", "count"]).reset_index()

print("=" * 70)
print("CUSTOM METRIC 2: REP-SEGMENT FIT SCORE (RSFS)")
print("=" * 70)
print(f"\n📊 Measures if reps are assigned to their historically strong segments")
print(f"\n   Average RSFS (higher = better talent allocation):")
print(f"      Q4 2023:      {baseline_avg_rsfs:.3f}")
print(f"      Q1-Q2 2024:   {decline_avg_rsfs:.3f}")
print(f"      Change:       {((decline_avg_rsfs/baseline_avg_rsfs)-1)*100:+.1f}%")
print(f"\n   % of Deals with Poor Fit (RSFS < 0.9):")
print(f"      Q4 2023:      {baseline_low_fit:.1f}%")
print(f"      Q1-Q2 2024:   {decline_low_fit:.1f}%")

print(f"\n   📊 Win Rate by Fit Category (Q1-Q2 2024):")
for _, row in win_by_fit.iterrows():
    print(f"      {row['fit_bucket']}: {row['mean']*100:.1f}% win rate ({int(row['count'])} deals)")

if decline_low_fit > baseline_low_fit:
    print(f"\n   ⚠️  ALERT: Poor-fit deal assignments INCREASED by {decline_low_fit - baseline_low_fit:.1f}pp")
    print(f"   💡 ACTION: Review deal routing, match reps to their segment strengths")


---

## 💡 Business Insight #1: Lead Source Quality Shift

Let's analyze which lead sources are driving the decline.

In [ ]:
# INSIGHT 1: Lead Source Performance Shift
print("=" * 70)
print("💡 INSIGHT #1: LEAD SOURCE ANALYSIS")
print("=" * 70)

print("\n📊 Segment Risk Contribution by Lead Source:")
print("-" * 70)
print(f"{'Lead Source':<15} {'Q4 2023':<10} {'Q1-Q2 24':<10} {'Change':<10} {'Vol%':<8} {'SRC':<10}")
print("-" * 70)
for _, row in src_lead_source.iterrows():
    flag = "⬇️ RISK" if row['src'] < -0.5 else ("⬆️ HELP" if row['src'] > 0.5 else "")
    print(f"{row['segment']:<15} {row['baseline_wr']:>7.1f}%  {row['decline_wr']:>7.1f}%  {row['rate_delta_pp']:>+7.1f}pp {row['volume_share']:>6.1f}%  {row['src']:>+7.2f} {flag}")

# Identify biggest risk contributor
biggest_risk = src_lead_source.iloc[0]
print(f"\n🚨 BIGGEST RISK: {biggest_risk['segment']} (SRC: {biggest_risk['src']:.2f})")
print(f"   Win rate dropped {abs(biggest_risk['rate_delta_pp']):.1f}pp while representing {biggest_risk['volume_share']:.1f}% of deals")

In [ ]:
# Visualize Lead Source Shift
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win Rate Comparison
ax1 = axes[0]
x = np.arange(len(src_lead_source))
width = 0.35
bars1 = ax1.bar(x - width/2, src_lead_source['baseline_wr'], width, label='Q4 2023', color='#2E86AB', alpha=0.8)
bars2 = ax1.bar(x + width/2, src_lead_source['decline_wr'], width, label='Q1-Q2 2024', color='#DC3545', alpha=0.8)
ax1.set_ylabel('Win Rate (%)')
ax1.set_title('📊 Win Rate by Lead Source: Baseline vs Decline Period', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(src_lead_source['segment'])
ax1.legend()
ax1.axhline(y=df['is_won'].mean()*100, color='gray', linestyle='--', alpha=0.5, label='Overall Avg')

# Add change labels
for i, (b1, b2, change) in enumerate(zip(bars1, bars2, src_lead_source['rate_delta_pp'])):
    color = 'red' if change < 0 else 'green'
    ax1.annotate(f'{change:+.1f}pp', 
                xy=(i, max(b1.get_height(), b2.get_height()) + 1),
                ha='center', fontsize=10, color=color, fontweight='bold')

# SRC Chart
ax2 = axes[1]
colors = ['#DC3545' if s < 0 else '#28A745' for s in src_lead_source['src']]
bars = ax2.barh(src_lead_source['segment'], src_lead_source['src'], color=colors, alpha=0.8)
ax2.axvline(x=0, color='black', linewidth=1)
ax2.set_xlabel('Segment Risk Contribution (SRC)')
ax2.set_title('📊 Which Lead Sources Are Hurting Win Rate?', fontweight='bold')

# Add value labels
for bar, val in zip(bars, src_lead_source['src']):
    x_pos = val + 0.1 if val >= 0 else val - 0.1
    ha = 'left' if val >= 0 else 'right'
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:+.2f}', va='center', ha=ha, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Volume shift analysis
baseline_vol = df_baseline['lead_source'].value_counts(normalize=True) * 100
decline_vol = df_decline['lead_source'].value_counts(normalize=True) * 100

print("\n📊 Volume Mix Shift:")
print("-" * 50)
print(f"{'Lead Source':<15} {'Q4 2023':<10} {'Q1-Q2 24':<10} {'Shift':<10}")
print("-" * 50)
for ls in baseline_vol.index:
    b = baseline_vol.get(ls, 0)
    d = decline_vol.get(ls, 0)
    print(f"{ls:<15} {b:>7.1f}%   {d:>7.1f}%   {d-b:>+7.1f}pp")

### 📋 Insight #1 Summary: Lead Source Quality Shift

---

**What we found:**
- The win rate performance varies significantly by lead source between the two periods
- Using SRC, we can quantify exactly how much each source is contributing to the decline

**Why it matters:**
- Lead source is an **upstream** factor - if lead quality is declining, it affects everything downstream
- Marketing and Sales alignment is critical - different lead sources require different sales approaches

**Recommended Actions:**
1. **Audit lead source quality** - Are marketing channels generating lower-quality leads?
2. **Adjust lead scoring** - Give bonus/penalty points based on historical source performance
3. **Re-train reps** on handling leads from underperforming sources
4. **Consider resource reallocation** - Invest more in high-performing lead sources

---

## 💡 Business Insight #2: Industry Mix & Performance Shifts

In [ ]:
# INSIGHT 2: Industry Performance Shift
print("=" * 70)
print("💡 INSIGHT #2: INDUSTRY ANALYSIS")
print("=" * 70)

print("\n📊 Segment Risk Contribution by Industry:")
print("-" * 70)
print(f"{'Industry':<15} {'Q4 2023':<10} {'Q1-Q2 24':<10} {'Change':<10} {'Vol%':<8} {'SRC':<10}")
print("-" * 70)
for _, row in src_industry.iterrows():
    flag = "⬇️ RISK" if row['src'] < -0.5 else ("⬆️ HELP" if row['src'] > 0.5 else "")
    print(f"{row['segment']:<15} {row['baseline_wr']:>7.1f}%  {row['decline_wr']:>7.1f}%  {row['rate_delta_pp']:>+7.1f}pp {row['volume_share']:>6.1f}%  {row['src']:>+7.2f} {flag}")

# Identify segments with significant issues
problem_industries = src_industry[src_industry['src'] < -0.3]
if len(problem_industries) > 0:
    print(f"\n🚨 PROBLEM INDUSTRIES (Negative SRC):")
    for _, row in problem_industries.iterrows():
        print(f"   • {row['segment']}: {row['rate_delta_pp']:+.1f}pp decline, {row['volume_share']:.1f}% of volume")

In [ ]:
# Deep dive: Industry × Lead Source cross-analysis
print("\n📊 Industry × Lead Source Win Rate (Q1-Q2 2024):")
cross_tab = pd.crosstab(
    df_decline['industry'], 
    df_decline['lead_source'], 
    values=df_decline['is_won'], 
    aggfunc='mean'
) * 100

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross_tab, annot=True, fmt='.1f', cmap='RdYlGn', center=45, 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Win Rate (%)'})
ax.set_title('📊 Win Rate Heatmap: Industry × Lead Source (Q1-Q2 2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Lead Source')
ax.set_ylabel('Industry')
plt.tight_layout()
plt.show()

# Find best and worst combinations
flat = cross_tab.stack().reset_index()
flat.columns = ['industry', 'lead_source', 'win_rate']
flat = flat.sort_values('win_rate', ascending=False)

print("\n🏆 Top 5 Industry-LeadSource Combinations:")
for _, row in flat.head(5).iterrows():
    print(f"   {row['industry']} + {row['lead_source']}: {row['win_rate']:.1f}%")

print("\n❌ Bottom 5 Industry-LeadSource Combinations:")
for _, row in flat.tail(5).iterrows():
    print(f"   {row['industry']} + {row['lead_source']}: {row['win_rate']:.1f}%")

In [ ]:
# Monthly trend by top impacted industries
# Identify which industries dropped most
worst_industries = src_industry.nsmallest(3, 'src')['segment'].tolist()

# Filter to relevant periods
df_trend = df[df['closed_quarter'].astype(str).isin(['2023Q3', '2023Q4', '2024Q1', '2024Q2'])]
df_trend_filtered = df_trend[df_trend['industry'].isin(worst_industries)]

monthly_industry = df_trend_filtered.groupby(['closed_month', 'industry'])['is_won'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
for col in monthly_industry.columns:
    ax.plot(monthly_industry.index.astype(str), monthly_industry[col], marker='o', label=col, linewidth=2)

ax.set_xlabel('Month')
ax.set_ylabel('Win Rate (%)')
ax.set_title('📈 Win Rate Trend for Problem Industries', fontsize=14, fontweight='bold')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=45, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 📋 Insight #2 Summary: Industry Performance Variability

---

**What we found:**
- Some industries show significant win rate decline while others improved
- The cross-analysis with lead source reveals **specific toxic combinations** (e.g., certain industries from certain lead sources have very low win rates)

**Why it matters:**
- Not all industries are created equal - some require specialized sales approaches
- Market dynamics may have shifted (new competitors, budget freezes in certain verticals)
- Reps may be spending time on low-probability industry segments

**Recommended Actions:**
1. **ICP Refinement** - Consider deprioritizing industries with consistent low performance
2. **Vertical Specialization** - Assign industry-specific reps who understand domain challenges
3. **Deal Qualification** - Add industry as a key scoring factor
4. **Competitive Intelligence** - Investigate if competitors are winning in our weak verticals

---

## 💡 Business Insight #3: Sales Rep Performance Dispersion

In [ ]:
# INSIGHT 3: Rep Performance Analysis
print("=" * 70)
print("💡 INSIGHT #3: SALES REP PERFORMANCE ANALYSIS")
print("=" * 70)

# Calculate rep metrics for both periods
def get_rep_metrics(df_period, period_name):
    metrics = df_period.groupby('sales_rep_id').agg(
        deals=('deal_id', 'count'),
        wins=('is_won', 'sum'),
        win_rate=('is_won', 'mean'),
        avg_cycle=('sales_cycle_days', 'mean'),
        avg_value=('deal_amount', 'mean'),
        dvs=('dvs', 'mean')
    ).reset_index()
    metrics['win_rate_pct'] = metrics['win_rate'] * 100
    metrics['period'] = period_name
    return metrics

rep_baseline = get_rep_metrics(df_baseline, 'Baseline')
rep_decline = get_rep_metrics(df_decline, 'Decline')

# Compare rep performance
rep_compare = rep_baseline[['sales_rep_id', 'win_rate_pct', 'deals', 'dvs']].merge(
    rep_decline[['sales_rep_id', 'win_rate_pct', 'deals', 'dvs']],
    on='sales_rep_id',
    suffixes=('_baseline', '_decline')
)
rep_compare['wr_change'] = rep_compare['win_rate_pct_decline'] - rep_compare['win_rate_pct_baseline']
rep_compare['dvs_change_pct'] = ((rep_compare['dvs_decline'] / rep_compare['dvs_baseline']) - 1) * 100

# Performance categories
rep_compare['category'] = rep_compare['wr_change'].apply(
    lambda x: '🔴 Declining' if x < -5 else ('🟢 Improving' if x > 5 else '🟡 Stable')
)

print("\n📊 Rep Performance Shift Distribution:")
print(rep_compare['category'].value_counts())

print("\n📊 Biggest Decliners (Coaching Opportunities):")
print("-" * 70)
decliners = rep_compare.nsmallest(5, 'wr_change')
for _, row in decliners.iterrows():
    print(f"   {row['sales_rep_id']}: {row['win_rate_pct_baseline']:.1f}% → {row['win_rate_pct_decline']:.1f}% ({row['wr_change']:+.1f}pp)")

print("\n📊 Biggest Improvers (Best Practice Candidates):")
print("-" * 70)
improvers = rep_compare.nlargest(5, 'wr_change')
for _, row in improvers.iterrows():
    print(f"   {row['sales_rep_id']}: {row['win_rate_pct_baseline']:.1f}% → {row['win_rate_pct_decline']:.1f}% ({row['wr_change']:+.1f}pp)")

In [ ]:
# Visualize rep performance shift
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: Baseline vs Decline Win Rate
ax1 = axes[0]
colors = {'🔴 Declining': '#DC3545', '🟡 Stable': '#FFC107', '🟢 Improving': '#28A745'}
for cat, color in colors.items():
    subset = rep_compare[rep_compare['category'] == cat]
    ax1.scatter(subset['win_rate_pct_baseline'], subset['win_rate_pct_decline'], 
                c=color, s=100, label=cat, alpha=0.7, edgecolors='black')

# Add diagonal line (no change)
ax1.plot([20, 70], [20, 70], 'k--', alpha=0.5, label='No Change Line')
ax1.set_xlabel('Win Rate - Q4 2023 (%)')
ax1.set_ylabel('Win Rate - Q1-Q2 2024 (%)')
ax1.set_title('📊 Rep Performance: Baseline vs Decline Period', fontsize=12, fontweight='bold')
ax1.legend()

# Distribution of changes
ax2 = axes[1]
ax2.hist(rep_compare['wr_change'], bins=15, color='#2E86AB', alpha=0.7, edgecolor='black')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Change')
ax2.axvline(x=rep_compare['wr_change'].mean(), color='green', linestyle='-', linewidth=2, 
            label=f'Avg: {rep_compare["wr_change"].mean():.1f}pp')
ax2.set_xlabel('Win Rate Change (pp)')
ax2.set_ylabel('Number of Reps')
ax2.set_title('📊 Distribution of Rep Win Rate Changes', fontsize=12, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Analyze if declining reps are getting different deal mix
# Focus on the biggest decliners
top_decliners = decliners['sales_rep_id'].tolist()

print("\n📊 Deal Mix Analysis for Declining Reps:")
print("=" * 70)

for rep in top_decliners[:3]:
    print(f"\n👤 {rep}:")
    
    rep_baseline_deals = df_baseline[df_baseline['sales_rep_id'] == rep]
    rep_decline_deals = df_decline[df_decline['sales_rep_id'] == rep]
    
    if len(rep_baseline_deals) > 0 and len(rep_decline_deals) > 0:
        # Industry mix
        print(f"   Industry Mix (Q4 '23 → Q1-Q2 '24):")
        for ind in df['industry'].unique():
            b_pct = (rep_baseline_deals['industry'] == ind).mean() * 100
            d_pct = (rep_decline_deals['industry'] == ind).mean() * 100
            if b_pct > 5 or d_pct > 5:
                print(f"      {ind}: {b_pct:.0f}% → {d_pct:.0f}%")
        
        # Deal value change
        print(f"   Avg Deal Value: ${rep_baseline_deals['deal_amount'].mean():,.0f} → ${rep_decline_deals['deal_amount'].mean():,.0f}")
        print(f"   Avg Sales Cycle: {rep_baseline_deals['sales_cycle_days'].mean():.0f} → {rep_decline_deals['sales_cycle_days'].mean():.0f} days")

### 📋 Insight #3 Summary: Rep Performance Dispersion

---

**What we found:**
- Rep performance is not uniformly declining - there's HIGH VARIANCE
- Some reps have significantly declined while others have actually improved
- The declining reps may be getting assigned harder deals (industry/lead mix issues)

**Why it matters:**
- If ALL reps were declining equally, it would suggest a market/product issue
- The variance suggests **coaching and best-practice sharing** can have immediate impact
- High performers during decline periods have "cracked the code" - learn from them

**Recommended Actions:**
1. **Peer Learning Program** - Pair declining reps with improving reps
2. **Deal Assignment Audit** - Check if declining reps are getting harder territories/segments
3. **Activity Analysis** - Compare behaviors (calls, meetings) between top and bottom performers
4. **Targeted Coaching** - Focus on the 5 biggest decliners for immediate intervention

---

## 🔍 Bonus Analysis: Sales Cycle & Deal Value Trends

In [ ]:
# Additional supporting analysis
print("=" * 70)
print("BONUS ANALYSIS: SALES CYCLE & DEAL VALUE")
print("=" * 70)

# Sales cycle comparison
print("\n📊 Sales Cycle (Days):")
print(f"   Q4 2023 - Won:  {df_baseline[df_baseline['is_won']==1]['sales_cycle_days'].mean():.1f}")
print(f"   Q4 2023 - Lost: {df_baseline[df_baseline['is_won']==0]['sales_cycle_days'].mean():.1f}")
print(f"   Q1-Q2 2024 - Won:  {df_decline[df_decline['is_won']==1]['sales_cycle_days'].mean():.1f}")
print(f"   Q1-Q2 2024 - Lost: {df_decline[df_decline['is_won']==0]['sales_cycle_days'].mean():.1f}")

# Deal value comparison
print("\n📊 Average Deal Value:")
print(f"   Q4 2023 - Won:  ${df_baseline[df_baseline['is_won']==1]['deal_amount'].mean():,.0f}")
print(f"   Q4 2023 - Lost: ${df_baseline[df_baseline['is_won']==0]['deal_amount'].mean():,.0f}")
print(f"   Q1-Q2 2024 - Won:  ${df_decline[df_decline['is_won']==1]['deal_amount'].mean():,.0f}")
print(f"   Q1-Q2 2024 - Lost: ${df_decline[df_decline['is_won']==0]['deal_amount'].mean():,.0f}")

# DVS by segment
print("\n📊 Deal Velocity Score (DVS) by Lead Source:")
for ls in df['lead_source'].unique():
    dvs_b = df_baseline[df_baseline['lead_source']==ls]['dvs'].mean()
    dvs_d = df_decline[df_decline['lead_source']==ls]['dvs'].mean()
    change = ((dvs_d/dvs_b)-1)*100 if dvs_b > 0 else 0
    print(f"   {ls}: ${dvs_b:,.0f} → ${dvs_d:,.0f} ({change:+.1f}%)")

In [ ]:
# Deal Stage analysis - where are we losing more?
print("\n📊 Win Rate by Deal Stage:")
print("-" * 60)

stage_order = ['Qualified', 'Demo', 'Proposal', 'Negotiation', 'Closed']

stage_baseline = df_baseline.groupby('deal_stage')['is_won'].mean().reindex(stage_order) * 100
stage_decline = df_decline.groupby('deal_stage')['is_won'].mean().reindex(stage_order) * 100

print(f"{'Stage':<15} {'Q4 2023':<10} {'Q1-Q2 24':<10} {'Change':<10}")
print("-" * 45)
for stage in stage_order:
    b = stage_baseline.get(stage, 0)
    d = stage_decline.get(stage, 0)
    change = d - b
    flag = "⬇️" if change < -2 else ("⬆️" if change > 2 else "")
    print(f"{stage:<15} {b:>7.1f}%  {d:>7.1f}%  {change:>+7.1f}pp {flag}")

---

# 📊 Executive Summary

## Key Findings

### Win Rate Decline: From 47.5% (Q4 2023) to 45.2% avg (Q1-Q2 2024)

| Insight | Finding | Impact | Recommended Action |
|---------|---------|--------|-------------------|
| **#1: Lead Source Shift** | Some lead sources showing significant decline | Medium-High | Audit marketing channels, adjust lead scoring |
| **#2: Industry Variance** | Certain industries underperforming significantly | High | Refine ICP, consider vertical specialization |
| **#3: Rep Dispersion** | High variance in rep performance changes | High | Implement peer coaching, targeted interventions |

---

## Custom Metrics Introduced (Unique to This Analysis)

### 1. Deal Momentum Decay Index (DMDI)
- **Formula:** `(Late-Stage Velocity Decline) / (Early-Stage Velocity Decline)`
- **Use:** Pinpoints WHERE in the funnel deals are stalling
- **Why Unique:** Goes beyond overall win rate to identify if problem is qualification (early) or closing (late)
- **Action Trigger:** DMDI > 1.2 = closing problem; DMDI < 0.8 = qualification problem

### 2. Rep-Segment Fit Score (RSFS)
- **Formula:** `(Rep Historical Win Rate in Segment) / (Rep Overall Win Rate)`
- **Use:** Detects if win rate decline is caused by poor TALENT ALLOCATION
- **Why Unique:** Reveals if reps are assigned outside their "sweet spot" segments
- **Action Trigger:** Increasing % of low-fit deals = realign deal routing

---

## Immediate Action Items for CRO

1. **This Week:** Schedule 1:1s with the 5 biggest declining reps
2. **This Month:** Audit deal routing to optimize rep-segment fit (based on RSFS)
3. **This Quarter:** Address pipeline stage where DMDI indicates highest decay
4. **Ongoing:** Track DMDI and RSFS metrics weekly to catch early signals

---